In [ ]:
# Install every third-party dependency into this notebook's active Python environment.
# OR-Tools includes the cross-platform GLOP linear-programming solver.
%pip install --quiet --disable-pip-version-check "ortools>=9.10" "pandas>=1.5"

In [ ]:
from ortools.linear_solver import pywraplp

In [ ]:
# Define the LP model using OR-Tools' bundled GLOP solver.
# GLOP is appropriate here because every decision variable is continuous.
SC = pywraplp.Solver.CreateSolver("GLOP")
if SC is None:
    raise RuntimeError("The OR-Tools GLOP solver is unavailable in this Python environment.")

In [ ]:
# product index
P = [1,2]
# operation index
O = [1,2]
# factories
F = [1,2,3]
# warehouses
W = [1,2,3,4,5]
# markets
M = [1,2,3,4,5,6,7,8,9]
# transport mode
MD = [1,2]

In [ ]:
# supplier RM capacity (product)
S = {1: 12, 2: 10.5}

In [ ]:
# shared production capacity of each operation (operation, product)
CAP = {(1,1): 5, (1,2): 5, (2,1): 8, (2,2): 4}

In [ ]:
# RM cost (product)
RC = {1: 4, 2: 3}

In [ ]:
# variable production cost (factory, product)
PC = {(1,1): 0.15, (1,2): 0.1, (2,1): 0.19, (2,2): 0.14, (3,1): 0.13, (3,2): 0.09}

In [ ]:
# transport cost - factory to market (factory, market)
TCFM = {(1,1): 100, (1,2): 0.6, (1,3): 100, (1,4): 100, (1,5): 100, (1,6): 100, (1,7): 100, (1,8): 100, (1,9): 100,
        (2,1): 100, (2,2): 100, (2,3): 100, (2,4): 100, (2,5): 100, (2,6): 100, (2,7): 100, (2,8): 100, (2,9): 100,
        (3,1): 100, (3,2): 100, (3,3): 100, (3,4): 100, (3,5): 100, (3,6): 100, (3,7): 0.5, (3,8): 0.6, (3,9): 100}

In [ ]:
# transport cost - factory to warehouse (factory, warehouse, transport mode)
TCFW = {(1,1,1): 1.1, (1,1,2): 100, (1,2,1): 0.7, (1,2,2): 0.5, (1,3,1): 0.6, (1,3,2): 100, (1,4,1): 0.7, (1,4,2): 100, (1,5,1): 0.9, (1,5,2): 0.7, 
       (2,1,1): 0.6, (2,1,2): 100, (2,2,1): 0.9, (2,2,2): 0.7, (2,3,1): 1.0, (2,3,2): 0.7, (2,4,1): 0.5, (2,4,2): 100, (2,5,1): 100, (2,5,2): 100,
       (3,1,1): 100, (3,1,2): 100, (3,2,1): 0.8, (3,2,2): 100, (3,3,1): 0.9, (3,3,2): 100, (3,4,1): 0.7, (3,4,2): 100, (3,5,1): 0.6, (3,5,2): 100}

In [ ]:
#transport cost - warehouse to market (warehouse, market) 
TCWM = {(1,1): 0.1, (1,2): 0.2, (1,3): 0.15, (1,4): 0.12, (1,5): 0.18, (1,6): 0.1, (1,7): 100, (1,8): 100, (1,9): 100,
       (2,1): 0.15, (2,2): 0.12, (2,3): 0.1, (2,4): 0.17, (2,5): 100, (2,6): 100, (2,7): 100, (2,8): 100, (2,9): 100,
       (3,1): 0.18, (3,2): 0.15, (3,3): 100, (3,4): 0.15, (3,5): 100, (3,6): 100, (3,7): 100, (3,8): 100, (3,9): 100,
       (4,1): 100, (4,2): 100, (4,3): 100, (4,4): 0.14, (4,5): 0.13, (4,6): 100, (4,7): 0.18, (4,8): 0.1, (4,9): 0.14,
       (5,1): 100, (5,2): 100, (5,3): 100, (5,4): 100, (5,5): 100, (5,6): 0.19, (5,7): 0.14, (5,8): 0.16, (5,9): 0.12}

In [ ]:
# demand (market, product)
DEM = {(1,1): 0.750, (1,2): 1.000, (2,1): 1.300, (2,2): 0, (3,1): 0.850, (3,2): 1.450, (4,1): 0.750, (4,2): 1.250,
      (5,1): 0.550, (5,2): 0.950, (6,1): 0.200, (6,2): 0.600, (7,1): 0.700, (7,2): 1.500, (8,1): 0.450, (8,2): 1.500,
      (9,1): 0.750, (9,2): 0.900}

In [ ]:
# sales price (market, product)
SP = {(1,1): 6.5, (1,2): 5.5, (2,1): 7, (2,2): 5.1, (3,1): 6.3, (3,2): 5, (4,1): 7.6, (4,2): 5,
      (5,1): 7.2, (5,2): 5.4, (6,1): 6.5, (6,2): 5.2, (7,1): 7.9, (7,2): 5.7, (8,1): 7.2, (8,2): 5,
      (9,1): 7, (9,2): 5.2}

In [ ]:
# cost of shortage
COST_SHORT = 100

In [ ]:
# define variables - factory to market flows
FM = {(p,f,m): SC.NumVar(0.0, SC.infinity(), f"FM_{p}_{f}_{m}") for p in P for f in F for m in M}
FM

In [ ]:
# define variables - factory to warehouse flows
FW = {(p,f,w,md): SC.NumVar(0.0, SC.infinity(), f"FW_{p}_{f}_{w}_{md}") for p in P for f in F for w in W for md in MD}
FW

In [ ]:
# define variables - warehouse to market flows
WM = {(p,w,m): SC.NumVar(0.0, SC.infinity(), f"WM_{p}_{w}_{m}") for p in P for w in W for m in M}
WM

In [ ]:
# define variables - shortage (unmet demand) (market, product)
SHORT = {(m,p): SC.NumVar(0.0, SC.infinity(), f"SHORT_{m}_{p}") for m in M for p in P}

In [ ]:
# add constraint: supplier RM limit
for p in P:
    SC.Add(sum(FW[p,f,w,md] for f in F for w in W for md in MD) + sum(FM[p,f,m] for f in F for m in M) <= S[p], f"RM_supplier_limit_{p}")

In [ ]:
# add constraint: production capacity
for f in F:
    for o in O:
        SC.Add(sum((sum(FW[p,f,w,md] for w in W for md in MD) + sum(FM[p,f,m] for m in M))/CAP[o,p] for p in P) <= 1, f"production_capacity_{f}_{o}")

In [ ]:
# add constraint: supply meets demand
for m in M:
    for p in P:
        SC.Add(sum(FM[p,f,m] for f in F) + sum(WM[p,w,m] for w in W) + SHORT[m,p] == DEM[m,p], f"demand_{m}_{p}")

In [ ]:
# add constraint: flow balance
for w in W:
    for p in P:
        SC.Add(sum(FW[p,f,w,md] for f in F for md in MD) == sum(WM[p,w,m] for m in M), f"flow_balance_{w}_{p}")

In [ ]:
# Add the objective function.
SC.Maximize(sum(SP[m,p]*(sum(FM[p,f,m] for f in F) + sum(WM[p,w,m] for w in W)) for m in M for p in P) - sum(RC[p]*(sum(FW[p,f,w,md] for f in F for w in W for md in MD) + sum(FM[p,f,m] for f in F for m in M)) for p in P) - sum(PC[f,p]*(sum(FW[p,f,w,md] for w in W for md in MD) + sum(FM[p,f,m] for m in M)) for f in F for p in P) - sum(TCFM[f,m]*FM[p,f,m] for p in P for f in F for m in M) - sum(TCFW[f,w,md]*FW[p,f,w,md] for p in P for f in F for w in W for md in MD) - sum(TCWM[w,m]*WM[p,w,m] for p in P for w in W for m in M) - sum(COST_SHORT*SHORT[m,p] for m in M for p in P))

In [ ]:
status = SC.Solve()
if status != pywraplp.Solver.OPTIMAL:
    raise RuntimeError(f"The solver did not find an optimal solution (status code: {status}).")
print("Solver status: OPTIMAL")

In [ ]:
for v in SC.variables():
    print(v.name(), v.solution_value())
print("Objective Fn Value =", SC.Objective().Value())

In [ ]:
import pandas as pd
activities = SC.ComputeConstraintActivities()
def constraint_slack(constraint):
    activity = activities[constraint.index()]
    slacks = []
    if constraint.lb() != -SC.infinity():
        slacks.append(activity - constraint.lb())
    if constraint.ub() != SC.infinity():
        slacks.append(constraint.ub() - activity)
    return min(slacks) if slacks else float("inf")
result_constraints = [{'name': c.name(), 'shadow price': c.dual_value(), 'slack': constraint_slack(c)}
     for c in SC.constraints()]
print(pd.DataFrame(result_constraints))

In [ ]:
result_variables = [{'name': v.name(), 'value': v.solution_value(), 'reduced cost': v.reduced_cost()}
     for v in SC.variables() if v.solution_value() >= 0.001]
print(pd.DataFrame(result_variables))

In [ ]:
result_variables = [{'name': v.name(), 'value': v.solution_value(), 'reduced cost': v.reduced_cost()}
     for v in SC.variables() if abs(v.solution_value()) < 1e-9]
print(pd.DataFrame(result_variables))

In [ ]:
for f in F:
    for p in P:
        X = sum(FM[p,f,m].solution_value() for m in M) + sum(FW[p,f,w,md].solution_value() for w in W for md in MD)
        print("Production at factory %g, product %g : " % (f,p), X)